In [1]:
import numpy as np
import pandas as pd
import json
import warnings

warnings.filterwarnings('ignore')

house_members_by_election = [
    # 1901, 1903, 1906, 
    # 1910, 1913, 1914, 1917, 1919, 
    # 1922, 1925, 1928, 1929,
    # 1931, 1934, 1937, 
    # 1940, 1943, 
    1946, 1949,
    1951, 1954, 1955, 1958, 
    1961, 1963, 1966, 1969,
    1972, 1974, 1975, 1977,
    1980, 1983, 1984, 1987,
    1990, 1993, 1996, 1998, 
    2001, 2004, 2007, 
    2010, 2013, 2016, 2019, 
    2022, 2025
]

party_to_colour = {
    'ALP': '#E50000',
    "ALPN": "#AF1818",
    'LP': '#0000FF', 
    'NP': "#1B7200",
    'CLP': '#0000FF',
    'CP': '#1B7200',
    'NPA': '#1B7200',
    'NAT': '#1B7200',
    'NCP': '#1B7200',
    'LNQ': "#0000FF",
    'LNP': '#0000FF',
    'GRN': "#00D600",
    'ON': '#FF8000',
    'KAP': "#804000",
    'UAP': "#CFD300",
    'PUP': "#CFD300",
    'IND': "#777777",
    'CA': "#777777",
    'NXT': "#777777",
    'XEN': "#777777",
}

labor = ['ALP', 'ALPN']
coalition =  ['LP', 'NP', 'CLP', 'LNQ', 'LNP', 'CP', 'NPA', 'NAT', 'NCP']

In [11]:
# Create colour-position data ByYear
for year in house_members_by_election:
    # Import data from year
    first_prefs_df = pd.read_csv(f"Data/Raw Data/First Prefs/first_prefs_{year}.csv")
    winners_df = pd.read_csv(f"Data/Raw Data/Winners/winner_{year}.csv")

    ### Create colour-position data

    # Add colour column to winners_df based on party
    winners_df["colour"] = winners_df["Party"].apply(lambda x: party_to_colour[x] if x in party_to_colour else "#777777")

    # Drop unnecessary columns from winners_df
    winners_df.drop(columns=["State", "Winner", 'Party'], inplace=True)
    winners_df.set_index("Division", inplace=True)

    # Create dictionaries for coordinates and vote shares
    xy = {}
    abc = {}

    # loop through divisions in winners_df and calculate vote shares and coordinates
    for div in winners_df.index:
        div_prefs = first_prefs_df.groupby("Division").get_group(div).sort_values("Votes", ascending=False)

        div_prefs['Votes'] = div_prefs["Votes"].replace('Unopposed', 1)  # Replace 0 votes with NaN to avoid skewing calculations
        div_prefs["Votes"] = pd.to_numeric(div_prefs["Votes"], errors='coerce').fillna(0)  # Convert to numeric, set non-convertible to 0

        total = div_prefs["Votes"].sum()
        n_votes = {row["Party"]: row["Votes"] for _, row in div_prefs.iterrows()}

        a = round(sum([n_votes.get(party, 0) for party in labor]) / total, 4)
        b = round(sum([n_votes.get(party, 0) for party in coalition]) / total, 4)
        c = round(1 - a - b, 4)

        abc[div] = [a, b, c]
        xy[div] = [round(float(np.sum(np.array([a, b, c])*[0, 1, 0.5])), 4), round(float(np.sum(np.array([a, b, c])*[0, 0, np.sqrt(3)/2])), 4)]

    
    # Save compact data for the year
    winners_df = winners_df.join(pd.DataFrame.from_dict(xy, orient='index', columns=['x', 'y']))
    winners_df = winners_df.join(pd.DataFrame.from_dict(abc, orient='index', columns=['a', 'b', 'c']))

    winners_df.to_json(f"Data/Compact Data/Colour-Positions/ByYear/{year}.json", orient='index', indent=2, index=True)

C:\Users\Adrien\AppData\Local\Temp\ipykernel_19288\2949391143.py:24: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  div_prefs['Votes'] = div_prefs["Votes"].replace('Unopposed', 1)  # Replace 0 votes with NaN to avoid skewing calculations
C:\Users\Adrien\AppData\Local\Temp\ipykernel_19288\2949391143.py:24: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  div_prefs['Votes'] = div_prefs["Votes"].replace('Unopposed', 1)  # Replace 0 votes with NaN to avoid skewing calculations
C:\Users\Adrien\AppData\Local\Temp\ipykernel_19288\2949391143.

In [13]:
# Create colour-position data ByDivision

all_divs = pd.read_csv('Data/Electorates.csv', usecols=["Electorate"])["Electorate"].values

for div in all_divs:
    data = {}
    for year in house_members_by_election:
        winners_df = pd.read_csv(f"Data/Raw Data/Winners/winner_{year}.csv")

        # Add colour column to winners_df based on party
        query = winners_df.query(f'Division == @div')
        if query.empty:
            continue

        data[year] = {}

        data[year]["colour"] = party_to_colour[query["Party"].iloc[0]]


        # Import data from year
        first_prefs_df = pd.read_csv(f"Data/Raw Data/First Prefs/first_prefs_{year}.csv")
        # Create dictionaries for coordinates and vote shares

        div_prefs = first_prefs_df.groupby("Division").get_group(div).sort_values("Votes", ascending=False)

        div_prefs['Votes'] = div_prefs["Votes"].replace('Unopposed', 1)  # Replace 0 votes with NaN to avoid skewing calculations
        div_prefs["Votes"] = pd.to_numeric(div_prefs["Votes"], errors='coerce').fillna(0)  # Convert to numeric, set non-convertible to 0

        total = div_prefs["Votes"].sum()
        n_votes = {row["Party"]: row["Votes"] for _, row in div_prefs.iterrows()}

        a = round(sum([n_votes.get(party, 0) for party in labor]) / total, 4)
        b = round(sum([n_votes.get(party, 0) for party in coalition]) / total, 4)
        c = round(1 - a - b, 4)

        data[year]["a"] = a
        data[year]['b'] = b
        data[year]['c'] = c

        data[year]['x'] = round(float(np.sum(np.array([a, b, c])*[0, 1, 0.5])), 4)
        data[year]['y'] = round(float(np.sum(np.array([a, b, c])*[0, 0, np.sqrt(3)/2])), 4)
        
    # Save compact data for the year
    with open(f"Data/Compact Data/Colour-Positions/ByDivision/{div}.json", 'w') as f:
        json.dump(data, f, indent=2)

In [1]:
import pandas as pd

df = pd.read_csv("Data/First Prefs/first_prefs_2022.csv")

In [15]:
df.groupby("Division").get_group("Adelaide").sort_values("Votes", ascending=False).set_index("Party").to_json("Data/Adelaide_2022.json", orient='index', indent=2, index=True)